# M2 · ②~⑤ 대시보드 데이터 전처리

**입력:** 공공데이터 ② 사고재해자수, ③ 사고사망자수, ④ 사업장 수, ⑤ 사망만인율. M1에서 API 응답과 다운로드 CSV가 일치한 파일을 사용한다. 네 자료 모두 포털 파일명 기준 2025-12-31 자료다. **출력:** 업종 × 규모 300개 조합의 `dashboard_metrics.parquet`, 자료별 긴 형식 파일, 전처리 검증 보고서.

원본은 바꾸지 않는다. 결측 14칸은 그대로 두고, 사고 건수·사업장 수·사망만인율의 단위를 섞지 않는다.

In [ ]:
from pathlib import Path
import json, re, unicodedata
import pandas as pd
import numpy as np
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists()), None)
assert ROOT is not None, 'SANUP-P 프로젝트에서 실행하세요.'
PERSONAL = ROOT/'data'/'personal'
(PERSONAL/'processed').mkdir(parents=True, exist_ok=True)
(PERSONAL/'reports').mkdir(parents=True, exist_ok=True)

## 1. 파일·열·키·결측 검사

## 원본 파일과 기준연도 확인

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd

ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
raw=ROOT/'data'/'personal'/'raw'
manifest=pd.read_csv(raw/'download_manifest.csv')
for source_id in (2,3,4,5):
    entry=manifest.loc[manifest.source_id.eq(source_id)]
    assert len(entry)==1, f'{source_id:02d} 원본 기록을 확인하세요.'
    expected=entry.iloc[0]
    path=raw/f'source_{source_id:02d}'/expected.filename
    actual=hashlib.sha256(path.read_bytes()).hexdigest().lower()
    assert actual==str(expected.sha256).lower(), f'{source_id:02d} 원본 파일이 수집 기록과 다릅니다.'
print('②~⑤ CSV 원본 해시 4개 확인')
# 기준일 2025-12-31은 CSV 열이 아니라 공공데이터포털의 각 파일명 끝 `_20251231`에서 확인했다.
REFERENCE_DATE='2025-12-31'

In [ ]:
SPECS = {
 '02': ('source_02_accident_injured.csv','accident_injured','명'),
 '03': ('source_03_accident_deaths.csv','accident_deaths','명'),
 '04': ('source_04_establishments.csv','establishments','개소'),
 '05': ('source_05_fatality_rate.csv','fatality_rate','공식 사망만인율'),
}
SIZE_CODES = ['lt5','5_9','10_19','20_29','30_49','50_99','100_299','300_499','500_999','1000_plus']
EXPECTED_SIZE_PREFIXES = ['5인미만','5인-9인','10인-19인','20인-29인','30인-49인',
                          '50인-99인','100인-299인','300인-499인','500인-999인','1000인이상']
def clean_category(value):
    return re.sub(r'\s+', '', unicodedata.normalize('NFKC', str(value))).strip()
def size_prefix(column):
    label=unicodedata.normalize('NFKC', str(column))
    label=re.sub(r'\s+', '', label)
    for dash in ('–','—','−','－'):
        label=label.replace(dash,'-')
    match=re.match(r'^(\d+인(?:미만|이상|-\d+인))',label)
    assert match, f'규모 구간을 열 이름에서 읽을 수 없습니다: {column}'
    return match.group(1)

frames={}
checks=[]
for source_id,(filename,metric,unit) in SPECS.items():
    path=PERSONAL/'raw'/f'source_{source_id}'/filename
    frame=pd.read_csv(path,encoding='cp949')
    assert len(frame.columns)==12, f'{source_id}: 기대한 업종 2열 + 규모 10열이 아닙니다.'
    assert list(frame.columns[:2])==['대업종','구분'], f'{source_id}: 업종 키 열 이름이 달라졌습니다.'
    keys=list(frame.columns[:2]); size_columns=list(frame.columns[2:])
    assert frame[keys].notna().all().all(), f'{source_id}: 업종 키 결측을 먼저 확인하세요.'
    prefixes=[size_prefix(column) for column in size_columns]
    assert prefixes==EXPECTED_SIZE_PREFIXES, f'{source_id}: 규모 구간 또는 열 순서 변경: {prefixes}'
    assert not frame.duplicated(subset=keys).any(), f'{source_id}: 업종 키 중복'
    assert len(frame)==30, f'{source_id}: 원본 행 수 변경. 새 파일인지 확인하세요.'
    for key in keys: frame[key]=frame[key].map(clean_category)
    assert not frame.duplicated(subset=keys).any(), f'{source_id}: 용어 정리 후 업종 키 중복'
    invalid=0
    for col in size_columns:
        before=frame[col].notna().sum()
        frame[col]=pd.to_numeric(frame[col],errors='coerce')
        invalid += int(before-frame[col].notna().sum())
    assert invalid==0, f'{source_id}: 숫자로 변환되지 않는 값 {invalid}개'
    values=frame[size_columns].to_numpy(dtype=float)
    assert (np.isfinite(values) | np.isnan(values)).all(), f'{source_id}: 무한대 수치 발견'
    assert not (frame[size_columns]<0).any().any(), f'{source_id}: 음수 발견'
    if source_id in {'02','03','04'}:
        finite=values[np.isfinite(values)]
        assert np.equal(finite, np.floor(finite)).all(), f'{source_id}: 건수·개소에 소수 발견'
    frames[source_id]=frame
    checks.append({'source_id':source_id,'rows':len(frame),'size_columns':size_columns,
                   'size_range_prefixes':prefixes,'schema_matches_expected':True,
                   'missing_cells':int(frame[size_columns].isna().sum().sum()),
                   'duplicate_industry_keys':0,'invalid_numeric_cells':invalid,'unit':unit})
display(pd.DataFrame([{k:v for k,v in c.items() if k!='size_columns'} for c in checks]))

## 2. 규모 열을 같은 코드로 펼치기

In [ ]:
long_frames={}
for source_id,(filename,metric,unit) in SPECS.items():
    frame=frames[source_id]
    keys=list(frame.columns[:2]); size_columns=list(frame.columns[2:])
    size_map=dict(zip(size_columns,SIZE_CODES,strict=True))
    long=frame.melt(id_vars=keys,value_vars=size_columns,var_name='source_size_label',value_name=metric)
    long=long.rename(columns={keys[0]:'industry_major',keys[1]:'industry_middle'})
    long['size_code']=long['source_size_label'].map(size_map)
    long['source_id']=source_id
    long['reference_date']=REFERENCE_DATE
    assert not long.duplicated(['industry_major','industry_middle','size_code']).any()
    assert len(long)==300
    path=PERSONAL/'processed'/f'source_{source_id}_{metric}_long.parquet'
    long.to_parquet(path,index=False)
    long_frames[source_id]=long
    print(source_id,len(long),'결측',int(long[metric].isna().sum()),'저장',path.name)

## 3. 공통 업종 × 규모 키로 결합하되, 각 지표 단위를 유지

In [ ]:
key=['industry_major','industry_middle','size_code']
dashboard=None
join_checks=[]
for source_id,(_,metric,unit) in SPECS.items():
    part=long_frames[source_id][key+[metric]]
    if dashboard is None:
        dashboard=part.copy()
    else:
        dashboard=dashboard.merge(part,on=key,how='outer',validate='one_to_one',indicator=True)
        join_checks.append({'source_id':source_id,'matched':int(dashboard['_merge'].eq('both').sum()),
                            'left_only':int(dashboard['_merge'].eq('left_only').sum()),
                            'right_only':int(dashboard['_merge'].eq('right_only').sum())})
        dashboard=dashboard.drop(columns='_merge')
assert len(dashboard)==300 and all(j['left_only']==j['right_only']==0 for j in join_checks)
dashboard['reference_date']=REFERENCE_DATE
dashboard=dashboard.sort_values(key).reset_index(drop=True)
out=PERSONAL/'processed'/'dashboard_metrics.parquet'
dashboard.to_parquet(out,index=False)
display(pd.DataFrame(join_checks))
print('결합 후',len(dashboard),'행; 사망만인율 결측',int(dashboard.fatality_rate.isna().sum()))

## 4. 검증 결과 기록

In [ ]:
report={'inputs':checks,'join_checks':join_checks,'output_rows':len(dashboard),
        'output_columns':dashboard.columns.tolist(),
        'output_missing':dashboard.isna().sum().astype(int).to_dict(),
        'reference_date':REFERENCE_DATE,
        'rules':['CP949 읽기','업종 키 결측 사전 검사','업종 표기 NFKC·공백 정리','규모 열 이름의 구간·순서 검증 후 코드화',
                 '건수·사업장 수 정수 검사',
                 '숫자 변환 불가값은 실패 처리','공통 키 one_to_one 결합','결측 대체·이상치 삭제 없음']}
cross_source_flags={
    'positive_rate_zero_accident_deaths':int(((dashboard.fatality_rate>0)&(dashboard.accident_deaths==0)).sum()),
    'rate_gt_1000':int((dashboard.fatality_rate>1000).sum()),
    'zero_establishments_with_injured':int(((dashboard.establishments==0)&(dashboard.accident_injured>0)).sum()),
}
report['cross_source_flags']=cross_source_flags
flag_columns=['industry_major','industry_middle','size_code','accident_deaths','fatality_rate',
              'accident_injured','establishments']
report['flagged_rows']={
    'positive_rate_zero_accident_deaths':dashboard.loc[(dashboard.fatality_rate>0)&(dashboard.accident_deaths==0),flag_columns].to_dict('records'),
    'rate_gt_1000':dashboard.loc[dashboard.fatality_rate>1000,flag_columns].to_dict('records'),
    'zero_establishments_with_injured':dashboard.loc[(dashboard.establishments==0)&(dashboard.accident_injured>0),flag_columns].to_dict('records'),
}
display(pd.DataFrame([cross_source_flags]))
display(dashboard.loc[(dashboard.establishments==0)&(dashboard.accident_injured>0),
    ['industry_major','industry_middle','size_code','accident_injured','establishments']])
report_path=PERSONAL/'reports'/'dashboard_preprocessing.json'
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('정제본:',out,'검증 기록:',report_path)

## 전처리 결과와 판단 근거

### 이번 단계에서 한 일

②~⑤ 원본 CSV의 SHA-256을 수집 매니페스트와 대조하고, 업종 2열과 규모 10열 구조를 확인했다. 규모 구간은 열 이름에서 직접 읽고 lt5부터 1000_plus까지 예상 순서와 맞는지 검사한 다음 긴 형식으로 펼쳤다. 업종·규모 키의 중복과 결측, 숫자 변환 불가, 음수, 건수·사업장 수의 소수 여부를 확인하고 공통 키를 일대일 결합했다.

- 자료별 30행×10규모 구간, 즉 300칸을 만들었다. 네 자료의 결합 키 300개가 모두 맞았고, 결합 후 중복이나 미매칭 키가 없었다.
- 자료의 기준일은 원본 페이지 표시 기준 2025-12-31로 기록했다. 현재 CSV 자체에는 기준일 열이 없으므로, 새 자료를 받을 때 포털의 발행 기준일과 파일 해시를 함께 갱신해야 한다.
- 숫자 결측은 임의로 채우지 않았고 원본의 0도 그대로 뒀다. 값을 제거하거나 다른 지표로 다시 계산하지 않았다.

### 알게 된 점과 데이터에서 찾은 것

- ⑤ 사망만인율에는 결측 14칸이 있다. 포털은 공란을 재해율 산출 불가로 설명하므로 0으로 채우지 않는다. 공식 정의는 임금근로자 1만 명당 사망자 수다.
- ③은 사고사망자수이고 ⑤는 임금근로자 수를 분모로 한 사망만인율이다. 두 지표의 범위와 분모가 다르므로, ③이 0인데 ⑤가 양수인 70칸은 자동 오류로 고치지 않는다. 두 값을 같은 사망자 정의로 취급하지 않는다.
- ⑤가 1,000을 넘는 칸은 8개였고 모두 광업 × 석탄광업및채석업에 모여 있었다. 최대값은 8,333.33이다. 단위가 만인율인 공식 값이므로 임의로 이상치 제거하지 않는다. 작은 집단의 비율은 분모가 작으면 크게 흔들릴 수 있지만, 이 파일에는 실제 임금근로자 분모가 없어 원인을 확정할 수 없다.
- 광업 × 석탄광업및채석업 × 1000_plus는 ② 사고재해자 1명인데 ④ 사업장 수는 0개소다. 원본 네 자료를 각각 확인해 발견한 불일치라서 그대로 보존하고, 사업장당 사고율처럼 ④를 분모로 쓰는 계산에는 포함하지 않는다.
- 사업장 수는 사업장 규모를 설명하는 보조 지표다. 근로자 수가 아니므로 ③÷④를 ⑤ 사망만인율이라고 부르지 않는다.

### 다음 단계에서 사용할 것

M3 대시보드는 ② 사고재해자수·③ 사고사망자수·④ 사업장 수·⑤ 공식 사망만인율을 각각 단위가 드러나게 표시한다. ⑤의 공란은 자료 없음/산출 불가로 남기고, 0과 구분한다. 업종·규모 필터가 바뀌어도 네 지표의 기준일과 분모 설명을 같이 보여준다.

### 유의할 점

①~⑤는 동일 연도라고 해서 조사 범위까지 자동으로 같아지는 것은 아니다. 특히 사고 사망자수와 사망만인율을 한 분자로 놓고 비율을 재계산하지 않는다. 고위험 업종 순위를 보여줄 때는 원자료의 집계 단위와 분모가 없는 한 인과관계나 개인 사업장의 위험도로 확장하지 않는다.

### 재현 파일과 검증 기록

- 결합본: data/personal/processed/dashboard_metrics.parquet
- 자료별 긴 형식: data/personal/processed/source_02~05_*_long.parquet
- 행·결측·결합·이상 플래그 상세: data/personal/reports/dashboard_preprocessing.json
- 공식 기준: [② 사고재해자수](https://www.data.go.kr/data/15084672/fileData.do), [③ 사고사망자수](https://www.data.go.kr/data/15084674/fileData.do), [④ 사업장수](https://www.data.go.kr/data/15064487/fileData.do), [⑤ 사망만인율](https://www.data.go.kr/data/15064491/fileData.do)
